## 1. Verify Pre-installed Packages

In [ ]:
import importlib.metadata

packages = [
    "langchain",
    "langchain-core",
    "langgraph",
    "langchain-aws",
    "langchain-mcp-adapters",
    "mcp",
    "httpx",
    "boto3",
]

print("Pre-installed packages:")
print("-" * 50)
for pkg in packages:
    try:
        version = importlib.metadata.version(pkg)
        print(f"{pkg:30} {version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{pkg:30} NOT INSTALLED")

In [ ]:
# Install packages
%pip install -U langgraph langchain-aws langchain-mcp-adapters -q

## 2. Imports

In [ ]:
import asyncio
import concurrent.futures
import warnings
from datetime import timedelta

from langchain_aws import ChatBedrockConverse
from langchain_mcp_adapters.tools import load_mcp_tools

# Suppress known bug: https://github.com/langchain-ai/langgraph/issues/6404
warnings.filterwarnings("ignore", message="create_react_agent has been moved")
from langgraph.prebuilt import create_react_agent

from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

print("All imports successful!")

## 3. Configuration

Set `MODEL` and your MCP Gateway credentials.

In [ ]:
# Load configuration from CONFIG.txt
from dotenv import load_dotenv
import os

load_dotenv("../CONFIG.txt")

MODEL_ID = os.getenv("MODEL_ID")
REGION = os.getenv("REGION", "us-west-2")
GATEWAY_URL = os.getenv("MCP_GATEWAY_URL")
ACCESS_TOKEN = os.getenv("MCP_ACCESS_TOKEN")

print(f"Model:   {MODEL_ID}")
print(f"Region:  {REGION}")

# Validate gateway credentials
if not GATEWAY_URL or "your-" in GATEWAY_URL:
    print("\nWARNING: Set MCP_GATEWAY_URL in CONFIG.txt before running MCP cells")
elif not ACCESS_TOKEN or "your-" in ACCESS_TOKEN:
    print("\nWARNING: Set MCP_ACCESS_TOKEN in CONFIG.txt before running MCP cells")
else:
    print(f"Gateway: {GATEWAY_URL[:50]}...")
    print("Configuration OK!")

## 4. System Prompt

In [ ]:
SYSTEM_PROMPT = """You are a helpful Neo4j database assistant with access to tools that let you query a Neo4j graph database.

Your capabilities include:
- Retrieve the database schema to understand node labels, relationship types, and properties
- Execute read-only Cypher queries to answer questions about the data
- Do not execute any write Cypher queries

When answering questions about the database:
1. First retrieve the schema to understand the database structure
2. Formulate appropriate Cypher queries based on the actual schema
3. If a query returns no results, explain what you looked for and suggest alternatives
4. Format results in a clear, human-readable way
5. Cite the actual data returned in your response

Important Cypher notes:
- Use MATCH patterns that align with the actual schema
- For counting, use MATCH (n:Label) RETURN count(n)
- For listing items, add LIMIT to avoid overwhelming results
- Handle potential NULL values gracefully

Be concise but thorough in your responses."""

## 5. Initialize LLM

In [ ]:
llm = ChatBedrockConverse(
    model=MODEL_ID,
    region_name=REGION,
    temperature=0,
)

print(f"LLM initialized with {MODEL_ID}!")

## 6. Query Helper

In [ ]:
async def query_async(question: str) -> str:
    """Ask the agent a question about the Neo4j database."""
    headers = {"Authorization": f"Bearer {ACCESS_TOKEN}"}
    
    async with streamablehttp_client(
        GATEWAY_URL,
        headers,
        timeout=timedelta(seconds=120),
        terminate_on_close=False
    ) as (read_stream, write_stream, _):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            tools = await load_mcp_tools(session)
            
            agent = create_react_agent(
                model=llm,
                tools=tools,
                prompt=SYSTEM_PROMPT,
            )
            
            result = await agent.ainvoke({
                "messages": [{"role": "user", "content": question}]
            })
            
            messages = result.get("messages", [])
            if messages:
                last_msg = messages[-1]
                return getattr(last_msg, "content", str(last_msg))
            return "No response"


def _run_async(coro):
    """Run async code in a new event loop in a separate thread."""
    loop = asyncio.new_event_loop()
    try:
        return loop.run_until_complete(coro)
    finally:
        loop.close()


def query(question: str) -> str:
    """Ask the agent a question about the Neo4j database."""
    print("=" * 70)
    print(f"Q: {question}")
    print("=" * 70)
    
    # Run in a separate thread to avoid Jupyter's event loop conflicts
    with concurrent.futures.ThreadPoolExecutor() as executor:
        future = executor.submit(_run_async, query_async(question))
        answer = future.result(timeout=180)
    
    print(f"\nA: {answer}")
    return answer

## 7. Demo Queries

In [ ]:
_ = query("What is the database schema? Give me a brief summary.")

In [ ]:
_ = query("How many nodes are in the database by label?")

In [ ]:
_ = query("What types of relationships exist in the database?")

## 8. Your Queries

In [ ]:
_ = query("List 5 sample records from the most populated node type.")

In [ ]:
# Your custom query
# _ = query("Your question here")